### 测出这块卡的"天花板"

**目标**：得到四个参照数，后面所有模型数字都拿它们对照。

| 符号 | 含义 | 用在哪 |
|---|---|---|
| P_fp32 | fp32（TF32 关）matmul 平台吞吐 | fp32 基线的预期时间 |
| P_tf32 | fp32（TF32 开）matmul 平台吞吐 | 可选参考行 |
| P_bf16 | bf16 matmul 平台吞吐 | 混合精度的预期时间 |
| BW | 显存拷贝带宽 | optimizer step 等访存密集算子的预期 |

#### 0. 前置检查（共享纪律）

```bash
cd /data2/c220/hz/cs336
nvidia-smi          # 两卡显存应接近 0，且没有 vllm server
mkdir -p results
```

#### 1. 脚本 `scripts/matmul_peak.py`


#### 2. 运行

用 tmux 开两个窗格，一个监控，一个跑实验：

```bash
# 窗格 A：监控（-i 用的是物理卡号）
nvidia-smi -i 0 --query-gpu=timestamp,clocks.sm,power.draw,temperature.gpu,utilization.gpu \
  --format=csv -l 1 > results/clock_gpu0.csv

# 窗格 B：跑实验
CUDA_VISIBLE_DEVICES=0 uv run python scripts/matmul_peak.py --tag gpu0
```

跑完后停掉监控，再对 1 号卡重复一遍（监控里 `-i 1`，实验里 `CUDA_VISIBLE_DEVICES=1 ... --tag gpu1`）。

#### 3. 怎么读结果

- 看每种模式的 **n=8192 和 16384 的 median**。小 n 吞吐低是因为没喂满 GPU，平台值才是天花板。
- 预期顺序：bf16 > fp32(TF32 开) > fp32(TF32 关)。如果 TF32 开和关几乎一样，检查输出里 `float32_matmul_precision` 那行是否真的变了。
- 两张卡的平台值应接近。差距明显：检查有没有别的任务，或者 `clock_gpu*.csv` 里 SM 频率有没有掉。
- 平台值与 median 的差异：`max` 与 `median` 差距大，说明测量抖动或降频，以 median 为准并在表里注明。
- memcopy 的参照：规格书标称显存带宽约 896 GB/s，实测通常是它的一部分，以实测为准。

- 填好的表

| 项目 | 卡 0 | 卡 1 | 采用值 |
|---|---|---|---|
| P_fp32（TF32 关，n=8192 / 16384） | 32.84 / 33.04 | 33.69 / 33.93 | **33.4 TFLOPs/s** |
| P_tf32（TF32 开） | 48.84 / 49.74 | 48.72 / 49.57 | **49.2** |
| P_bf16 | 92.36 / 98.41 | 95.03 / 98.74 | **96.1**（±3%） |
| BW（memcopy，读+写） | 768.0 | 769.6 | **769 GB/s** |
| 功耗墙 | 300 W | 300 W | 300 W |
| 稳态 SM 频率（fp32 关 / TF32 开 / bf16） | 约 2540 / 2812 / 2795 MHz | 约2650/2800/2800 | 记录即可 |

倍数关系可以写进报告：bf16 / fp32 ≈ 2.9×，TF32 / fp32 ≈ 1.5×，bf16 / TF32 ≈ 2.0×。memcopy 769 GB/s 约为标称带宽 896 GB/s 的 86%（标称值请以官方规格为准）。

**可选的验证**：确认降频原因的话，可以在跑 fp32 阶段时执行 `nvidia-smi -q -d PERFORMANCE`，看 "Clocks Event Reasons" 里是否有 `SW Power Cap: Active`。

- 决定：fp32 基线 = TF32 关

后面所有"预期时间"使用：

| 精度 | P | 用于 |
|---|---|---|
| fp32 基线 | 33.4 TFLOPs/s | 3.2、3.3 的 fp32 部分 |
| bf16 autocast | 96.1 TFLOPs/s | 3.5 混合精度对比 |
| 访存 | 769 GB/s | optimizer step |

在 3.1 的"TF32 设置"栏填：`torch.set_float32_matmul_precision("highest")`（TF32 关），并在 `benchmark.py` 里显式设置。

- 立刻可以算出的预期值（Step 2 的落地）

用第 4 节的 FLOPs 除以上面的 P，**B=4，ctx=512，fp32，TF32 关**：

| 模型 | forward (ms) | backward (ms) | forward+backward (ms) |
|---|---|---|---|
| small | 16.0 | 32.0 | 48.0 |
| medium | 53.7 | 107.4 | 161.1 |
| large | 123 | 246 | 369 |

仅 forward 的大模型（B=4，ctx=512，no_grad）：

| 模型 | fp32 (ms) | bf16 (ms) |
|---|---|---|
| xl | 253 | 88 |
| 2.7B | 424（显存约 12.7 GiB，贴边，不建议） | 148 |

optimizer step 下限（28 B/参数 ÷ 769 GB/s）：small 约 4.7 ms，medium 约 15.4 ms。

**这些都是"理想上限"，也就是时间的下界**，实测应该更慢。原因是模型里的矩阵乘比 8192³ 小得多，效率达不到平台值。可以用你刚测的规模曲线估计：small 的 FFN matmul（2048×768 乘 768×3072）约 9.7 GFLOP，相当于 n≈1700 的方阵，对应曲线上 n=1024 到 2048 附近，fp32 约为平台值的 75% 到 90%，bf16 约为 80% 到 97%。所以：
- small 的 t_meas / t_pred 预期在 1.2 到 2 之间；
- 如果 small 在 ctx=128 时远高于 2，那是 kernel 发射开销，不是 matmul 慢。


## profile
### 1.用手算建立预期

一个训练步的 FLOPs 约为 **6 × 参数量（不含 embedding）× token 数**（forward 占 1/3，backward 占 2/3）。

例：small，batch=4，ctx=512。非 embedding 参数约 85M，token 数 = 2048，所以约 1 TFLOP/步。除以 Step 1 测出的实际吞吐，就是你"应该"看到的步时间。

实测和这个预期差 2 倍以内就算正常。差得很远，说明脚本有问题，或者被 launch 开销、非 matmul 算子拖慢，这正是后面要分析的内容。


#### 预测清单（

以下均为 small，$B=4$，$T=512$，fp32，单卡。

**(1) 阶段时间**

| 阶段 | FLOPs | $P$（TFLOP/s） | 理想时间 | 访存部分 | 预测总时间 |
|---|---|---|---|---|---|
| forward | 0.534 T | 33.4 | 16.0 ms | 约 20 ms | 约 37 ms |
| backward | 1.068 T | 33.4 | 32.0 ms | 取 forward 的约 2 倍作粗估 | 待估 |
| optimizer | 28 B/参数 | BW 769 GB/s | 4.7 ms（下限） | | 见下 |

optimizer 的 4.7 ms 是带宽下限。staff 的 AdamW 若是逐参数 Python 循环，实际会被 kernel 发射开销主导，预期明显高于下限（12.9 解释）。

**(2) 时间占比**（FLOPs 占比与预测时间占比并排）

| 项 | FLOPs 占比 | 预测时间占比 |
|---|---|---|
| 注意力（matmul + 缩放 + mask + softmax） | 7.2% | 约 35%（第 8 章） |
| FFN（GEMM + 逐元素） | 65.2% | 约 40% |
| 其余（投影、norm、RoPE、残差、`lm_head`、loss） | 约 27.6% | 约 25%（余量） |

**(3) forward kernel 数与时间，按类别**

| 类别 | 预测 kernel 数 | 预测时间占比 |
|---|---|---|
| gemm | 约 109（$(7L+1)$ 个线性层加 $2L$ 个注意力 matmul） | 约 45% |
| elementwise | 约 300 | 约 40% |
| reduce | 约 25 | 约 3% |
| softmax | 12 | 约 5% |
| copy / cat | 约 40 | 约 3% |

这张表是按第 8.9 的 kernel 清单粗略相加的，误差较大，实测后直接替换。

**(4) forward 时间，按区间**（沿用第 8.9、9.9）

| 区间 | 预测 kernel 数 | 预测时间 (ms) |
|---|---|---|
| embed | 1 | 0.02 |
| block（×12） | 约 600（每个约 50） | 34.8 |
| ln_final | 6 | 0.09 |
| lm_head | 1 | 0.94 |
| loss | 数个 | 1.1 |

##### 显存预估

| 部分 | 估算 |
|---|---|
| 权重 + 梯度 + Adam | $16P$ B（fp32，$P$ 为参数个数） |
| FFN 激活 | $5Nd_{ff}$ 每层 |
| 注意力分数 | $c\,BhT^2$ 每层，$c\approx1$ 到 3 |
| 其余 $N\times d$ 张量 | 约 15 到 25 份每层 |
| logits 与损失 | $NV$ 的约 3 到 4 份 |

```python
def est_peak_gib(P, d, L, h, V, B, T, c=2, other_per_layer=20):
    N = B * T
    static = 16 * P
    act = L * 4 * (5 * N * 4 * d + c * B * h * T * T + other_per_layer * N * d) + 4 * 4 * N * V
    return (static + act) / 2**30
```



### 2.`scripts/benchmark.py`测量

- 详见benchmark.ipynb


### 3.2 端到端计时
固定 warmup=5、测量 20 步。context 取 128 / 256 / 512 / 1024。

| 模型 | ctx | forward 均值±std (ms) | backward 均值±std (ms) | optimizer step 均值±std (ms) | 状态 |
|---|---|---|---|---|---|
| small | 128 |  |  |  |  |
| small | 256 |  |  |  |  |
| small | 512 |  |  |  |  |
| small | 1024 |  |  |  |  |
| medium | 128 |  |  |  |  |
| medium | 256 |  |  |  |  |
| medium | 512 |  |  |  |  |
| medium | 1024 |  |  |  |  |
| large | 128 |  |  |  |  |
| large | 256 |  |  |  |  |
| large | 512 |  |  |  |  |
| large | 1024 |  |  |  |  |
| xl（仅 fwd） | 128 |  | N/A | N/A |  |
| 2.7B（仅 fwd, bf16） | 128 |  | N/A | N/A |  |

状态列填：OK / OOM / 降配（写明降成什么）。
预填预期：xl 仅 fwd fp32 约 4×4B=16GB 超显存 → 需降配或 bf16；2.7B bf16
参数 ~5.4GB，ctx=128 激活可控，预期 OK。实测后修正。
**需要回答**：
- 哪个阶段最慢？随 ctx 如何变化？
- 测量方差大不大？为什么？


```bash
 CUDA_VISIBLE_DEVICES=0 uv run python scripts/benchmark_v2.py \
      --model small medium large --ctx 128 256 512 1024 --warmup 5 --iters 10
device: NVIDIA GeForce RTX 5070 Ti  torch 2.11.0+cu130  cuda 13.0  git 55d7ad1
提示：运行前请用 nvidia-smi 确认卡空闲；抖动(jitter)过大先查是否有别的进程。

[small ctx=128 b=4 float32] params=128.6M  matmul_precision=highest  status=OK  loss0=9.2675
  phase                mean±std    median      min      max  jitter
  fwd(no_grad)       10.94±0.05     10.93    10.88    11.07     2%   11.55 TFLOP/s  peak 0.52 GiB
  phase                mean±std    median      min      max  jitter
  forward(+loss)     13.16±0.16     13.09    13.02    13.54     4%
  backward           20.00±0.23     20.04    19.68    20.34     3%
  optimizer          13.07±0.10     13.04    13.00    13.36     3%
  step=sum           46.23±0.34     46.17    45.79    46.73     2%
  e2e wall 45.49 ms/步  11254 tok/s  8.20 TFLOP/s(3xF, 按 step 中位数)  peak 反向后 2.19 / 整步 2.19 GiB  loss有限=True

[small ctx=256 b=4 float32] params=128.6M  matmul_precision=highest  status=OK  loss0=9.2774
  phase                mean±std    median      min      max  jitter
  fwd(no_grad)       16.33±0.06     16.31    16.28    16.48     1%   15.78 TFLOP/s  peak 0.56 GiB
  phase                mean±std    median      min      max  jitter
  forward(+loss)     16.99±0.05     16.98    16.94    17.09     1%
  backward           31.60±0.05     31.60    31.54    31.69     0%
  optimizer          12.98±0.05     13.00    12.89    13.03     1%
  step=sum           61.58±0.07     61.55    61.51    61.73     0%
  e2e wall 61.36 ms/步  16688 tok/s  12.54 TFLOP/s(3xF, 按 step 中位数)  peak 反向后 3.03 / 整步 3.03 GiB  loss有限=True

[small ctx=512 b=4 float32] params=128.6M  matmul_precision=highest  status=OK  loss0=9.2808
  phase                mean±std    median      min      max  jitter
  fwd(no_grad)       37.61±0.20     37.61    37.32    37.95     2%   14.20 TFLOP/s  peak 0.72 GiB
  phase                mean±std    median      min      max  jitter
  forward(+loss)     38.49±0.02     38.50    38.47    38.53     0%
  backward           76.09±0.04     76.10    76.01    76.13     0%
  optimizer          13.13±0.10     13.10    13.01    13.29     2%
  step=sum          127.71±0.11    127.68   127.54   127.88     0%
  e2e wall 127.53 ms/步  16059 tok/s  12.55 TFLOP/s(3xF, 按 step 中位数)  peak 反向后 5.04 / 整步 5.04 GiB  loss有限=True

[small ctx=1024 b=4 float32] params=128.6M  matmul_precision=highest  status=OK  loss0=9.2874
  phase                mean±std    median      min      max  jitter
  fwd(no_grad)       98.54±0.16     98.47    98.34    98.78     0%   11.63 TFLOP/s  peak 1.32 GiB
  phase                mean±std    median      min      max  jitter
  forward(+loss)    100.59±0.06    100.58   100.53   100.74     0%
  backward          198.40±0.06    198.37   198.33   198.51     0%
  optimizer          13.04±0.08     13.04    12.95    13.17     2%
  step=sum          312.02±0.11    312.00   311.91   312.27     0%
  e2e wall 311.97 ms/步  13129 tok/s  11.01 TFLOP/s(3xF, 按 step 中位数)  peak 反向后 10.85 / 整步 10.85 GiB  loss有限=True

[medium ctx=128 b=4 float32] params=423.2M  matmul_precision=highest  status=OK  loss0=9.3144
  phase                mean±std    median      min      max  jitter
  fwd(no_grad)       25.85±0.07     25.89    25.72    25.92     1%   16.58 TFLOP/s  peak 1.63 GiB
  phase                mean±std    median      min      max  jitter
  forward(+loss)     25.99±0.03     25.98    25.97    26.05     0%
  backward           51.28±0.07     51.28    51.13    51.37     0%
  optimizer          47.20±0.09     47.18    47.11    47.41     1%
  step=sum          124.47±0.15    124.47   124.20   124.77     0%
  e2e wall 124.61 ms/步  4109 tok/s  10.35 TFLOP/s(3xF, 按 step 中位数)  peak 反向后 6.43 / 整步 6.52 GiB  loss有限=True

[medium ctx=256 b=4 float32] params=423.2M  matmul_precision=highest  status=OK  loss0=9.3265
  phase                mean±std    median      min      max  jitter
  fwd(no_grad)       46.93±0.05     46.91    46.88    47.03     0%   18.57 TFLOP/s  peak 1.68 GiB
  phase                mean±std    median      min      max  jitter
  forward(+loss)     47.42±0.14     47.43    47.21    47.59     1%
  backward           93.38±0.05     93.39    93.28    93.44     0%
  optimizer          47.17±0.03     47.18    47.12    47.20     0%
  step=sum          187.98±0.14    187.99   187.78   188.23     0%
  e2e wall 187.78 ms/步  5453 tok/s  13.91 TFLOP/s(3xF, 按 step 中位数)  peak 反向后 8.49 / 整步 8.49 GiB  loss有限=True

[medium ctx=512 b=4 float32] params=423.2M  matmul_precision=highest  status=OK  loss0=9.3120
  phase                mean±std    median      min      max  jitter
  fwd(no_grad)      112.18±0.09    112.17   112.07   112.38     0%   16.00 TFLOP/s  peak 1.89 GiB
  phase                mean±std    median      min      max  jitter
  forward(+loss)    112.95±0.10    112.92   112.81   113.13     0%
  backward          218.83±0.45    218.67   218.48   220.01     1%
  optimizer          47.62±0.26     47.66    47.13    47.94     2%
  step=sum          379.40±0.53    379.29   378.85   380.77     1%
  e2e wall 379.40 ms/步  5398 tok/s  14.19 TFLOP/s(3xF, 按 step 中位数)  peak 反向后 13.72 / 整步 13.72 GiB  loss有限=True

[medium ctx=1024 b=4 float32] params=423.2M  matmul_precision=highest  status=OOM(train)  loss0=9.2988
  phase                mean±std    median      min      max  jitter
  fwd(no_grad)      290.96±0.23    291.08   290.48   291.19     0%   13.04 TFLOP/s  peak 2.69 GiB

[large ctx=128 b=4 float32] params=969.4M  matmul_precision=highest  status=OOM(train)  loss0=9.3158
  phase                mean±std    median      min      max  jitter
  fwd(no_grad)       58.75±0.02     58.75    58.72    58.79     0%   16.88 TFLOP/s  peak 3.77 GiB

[large ctx=256 b=4 float32] params=969.4M  matmul_precision=highest  status=OOM(train)  loss0=9.3082
  phase                mean±std    median      min      max  jitter
  fwd(no_grad)      108.35±0.06    108.34   108.30   108.50     0%   18.53 TFLOP/s  peak 3.84 GiB

[large ctx=512 b=4 float32] params=969.4M  matmul_precision=highest  status=OOM(train)  loss0=9.3173
  phase                mean±std    median      min      max  jitter
  fwd(no_grad)      255.48±0.34    255.30   255.20   256.24     0%   16.10 TFLOP/s  peak 4.10 GiB

[large ctx=1024 b=4 float32] params=969.4M  matmul_precision=highest  status=OOM(train)  loss0=9.3238
  phase                mean±std    median      min      max  jitter
  fwd(no_grad)      630.42±0.22    630.48   630.00   630.69     0%   13.65 TFLOP/s  peak 5.10 GiB

```

### 3.2 端到端计时（ms，均值±样本std）
| 模型 | ctx | dtype | forward 均值±std (ms) | backward 均值±std (ms) | optimizer step 均值±std (ms) | 状态 |
|---|---|---|---|---|---|---|
| small | 128 | fp32 | 13.16±0.16 | 20.00±0.23 | 13.07±0.10 | OK |
| small | 256 | fp32 | 16.99±0.05 | 31.60±0.05 | 12.98±0.05 | OK |
| small | 512 | fp32 | 38.49±0.02 | 76.09±0.04 | 13.13±0.10 | OK |
| small | 1024 | fp32 | 100.59±0.06 | 198.40±0.06 | 13.04±0.08 | OK |
| medium | 128 | fp32 | 25.99±0.03 | 51.28±0.07 | 47.20±0.09 | OK |
| medium | 256 | fp32 | 47.42±0.14 | 93.38±0.05 | 47.17±0.03 | OK |
| medium | 512 | fp32 | 112.95±0.10 | 218.83±0.45 | 47.62±0.26 | OK |
| medium | 1024 | fp32 | 290.96±0.23* | N/A | N/A | OOM(train) |
| large | 128 | fp32 | 58.75±0.02* | N/A | N/A | OOM(train) |
| large | 256 | fp32 | 108.35±0.06* | N/A | N/A | OOM(train) |
| large | 512 | fp32 | 255.48±0.34* | N/A | N/A | OOM(train) |
| large | 1024 | fp32 | 630.42±0.22* | N/A | N/A | OOM(train) |

说明：带 * 的是 no_grad 前向；其余 forward 是训练中的前向（含 loss、带计算图）。

saved -> results/bench/bench2_small_medium_large_ctx128-256-512-1024_b4_float32.json

### 3.3 warmup 影响
模型固定为 small，ctx=256。
| warmup 步数 | forward 均值±std (ms) | forward+backward 均值±std (ms) |
|---|---|---|
| 0 |  |  |
| 1 |  |  |
| 2 |  |  |
| 5 |  |  |
| 10 |  |  |
**需要回答**：没有 warmup 时为什么慢且方差大？
（候选因素：CUDA context 初始化、kernel/JIT 编译、cudnn autotune、
allocator 首次分配与扩容、L2 cache 冷启动——按本机实测归因）

```bash
CUDA_VISIBLE_DEVICES=0 uv run python scripts/benchmark_v2.py --model small --ctx 256 --dtype float32 --warmup 0 1 2 5 10 --iters 5
```

### 3.nsys 测量

三个阶段的最长 kernel、占比、调用次数都和之前一致，表可以填了。但这次的毫秒数有问题，**表里只写占比和次数**

#### 填表

| 模型 / ctx | 阶段 | 耗时最长 kernel | 占该阶段 % | 调用次数 | 是否 GEMM |
|---|---|---|---|---|---|
| small / 512 | forward | cuBLASLt fp32 GEMM（`magma_sgemmEx_kernel`） | 47.5% | 85 / 步 | 是 |
| small / 512 | backward | `cutlass_80_simt_sgemm_128x64_8x5_nt` | 14.6% | 72 / 步 | 是 |
| small / 512 | optimizer | `vectorized_elementwise_kernel`（乘法） | 44.5% | 666 / 步 | 否 |


| 问题 | 答案 |
|---|---|
| forward 中占比最高的 kernel？ | `magma_sgemmEx_kernel`，47.5%。73 次是 Linear，12 次是 QKᵀ |
| 同一 kernel 在 fwd+bwd 中占比变化？ | 47.5% → 16.6%。backward 里它只剩 12 次，GEMM 换成了 cutlass 的多种 tile。GEMM 整类：65.7% → 57.4% → 整步 53.5% |
| 除 matmul 外占时间最多的 kernel？ | 单个 kernel 是 vectorized elementwise 乘法，占整步 8.7%；按类别，elementwise 占整步 43.2% |
| 一次完整训练步中 matmul 占比？ | 53.5% |
| attention 内 softmax 与 matmul 的耗时比，与 FLOPs 比相比如何？ | 耗时比 2.15 : 1，FLOPs 比 0.020 : 1。按单位 FLOPs 的耗时算，softmax 比 matmul 高约 100 倍量级 |

三点注释：

- backward 第一名只有 14.6%，是因为同类 GEMM 按 tile 配置被拆成几个名字（`128x64 nt` 72 次、`128x64 nn` 60 次、`256x128 nn` 25 次）。我用上一次输出和这次的差值推算，backward 的 GEMM 整类约占 53%。
- optimizer 的 666 次是 111 的倍数，说明 111 个参数张量逐个发射，每个参数的同类 kernel 出现 6 次。
- Q5 的 matmul 指 QKᵀ 加 PV；FLOPs 里 softmax 按每元素 5 次运算估计，matmul 没有按因果 mask 减半。结论对这两个常数不敏感。


#### 结论

- **forward**：时间由 GEMM 主导（65.7%），最大的单项是 cuBLASLt 的 fp32 GEMM，覆盖 Linear 和 QKᵀ。
- **backward**：GEMM 仍占约一半，但分散在多种 cutlass tile 里；elementwise 个数是 forward 的约 2 倍，单次耗时更长。
- **optimizer**：全是 elementwise，且是逐参数发射（111 的倍数），瓶颈在发射开销而非计算，应换融合或 foreach 版本。
- **整步**：GEMM 53.5%，elementwise 43.2%，两类合计约 97%。
- **attention**：softmax 的 FLOPs 只有 matmul 的 2%，耗时却是 2 倍多，原因是 softmax 是由 5 个 kernel 拼成的访存密集运算（ncu 里 DRAM 约 83%）。

```bash
CUDA_VISIBLE_DEVICES=0 uv run nsys profile \
    -o results/nsys/small_ctx512_fp32 --force-overwrite=true \
    --trace=cuda,nvtx,osrt \
    --capture-range=cudaProfilerApi --capture-range-end=stop \
    python scripts/profile_run.py --model small --ctx 512 --batch 4 \
        --mode full --warmup 5 --steps 3 --nvtx-fine
```

- 导出为sqlite

```bash
nsys export --type sqlite --force-overwrite true \
  -o results/nsys/small_ctx512_fp32.sqlite results/nsys/small_ctx512_fp32.nsys-rep
```

- sql查询

In [ ]:
#!/usr/bin/env python
"""按 NVTX 窗口归因 kernel，产出三阶段 top-1 kernel 表。"""
import sqlite3
import pandas as pd

DB = "../results/nsys/small_ctx512_fp32.sqlite"
STEPS = 3  # 与 --steps 一致

db = sqlite3.connect(DB)

# 1) 全部 kernel（名字在 StringIds 表里；旧版 nsys 字段叫 shortName）
kern = pd.read_sql("""
    SELECT k.start, k.end, (k.end - k.start) AS dur_ns,
           s.value AS name
    FROM CUPTI_ACTIVITY_KIND_KERNEL k
    JOIN StringIds s ON k.demangledName = s.id
""", db)

# 2) 三个 phase 的 NVTX 窗口（每步各一个，全部取出）
win = pd.read_sql("""
    SELECT start, end, text AS phase
    FROM NVTX_EVENTS
    WHERE text IN ('forward', 'backward', 'optimizer')
""", db)

# 3) kernel 落窗归因：phase() 有 sync，包含关系是干净的
rows = []
for _, w in win.iterrows():
    m = kern[(kern.start >= w.start) & (kern.end <= w.end)].copy()
    m["phase"] = w.phase
    rows.append(m)
df = pd.concat(rows, ignore_index=True)

# 4) 按 阶段×kernel名 聚合
agg = (df.groupby(["phase", "name"])
         .agg(count=("dur_ns", "size"), total_ns=("dur_ns", "sum"))
         .reset_index())

def is_gemm(name: str) -> bool:
    n = name.lower()
    return any(k in n for k in ("gemm", "syrk", "cutlass", "magma"))


# 5) 出表
print(f"{'阶段':12s} {'最长kernel(截断)':52s} {'占阶段%':>7s} {'次/步':>7s} GEMM")
for phase in ["forward", "backward", "optimizer"]:
    sub = agg[agg.phase == phase].sort_values("total_ns", ascending=False)
    phase_total = sub.total_ns.sum()
    top = sub.iloc[0]
    name = top["name"]
    name = (name[:49] + "...") if len(name) > 52 else name
    print(f"{phase:12s} {name:52s} {100*top.total_ns/phase_total:6.1f}% "
          f"{top['count']/STEPS:7.1f} {'是' if is_gemm(name) else '否'}")
    # 顺手打印 top5 供交叉核对
    for _, r in sub.iloc[1:5].iterrows():
        n = r["name"][:49]
        print(f"{'':12s}   {n:50s} {100*r.total_ns/phase_total:6.1f}% {r['count']/STEPS:7.1f}")


阶段           最长kernel(截断)                                            占阶段%     次/步 GEMM
forward      void magma_sgemmEx_kernel<float, float, float, (b...   47.5%    85.0 是
               void cutlass::Kernel2<cutlass_80_simt_sgemm_128x2    14.8%    12.0
               void at::native::elementwise_kernel<(int)128, (in     5.0%    12.0
               void at::native::vectorized_elementwise_kernel<(i     4.5%    12.0
               void at::native::elementwise_kernel<(int)128, (in     4.0%    12.0
backward     void cutlass::Kernel2<cutlass_80_simt_sgemm_128x6...   14.6%    72.0 是
               void cutlass::Kernel2<cutlass_80_simt_sgemm_128x6    13.6%    60.0
               void cutlass::Kernel2<cutlass_80_simt_sgemm_256x1    13.2%    25.0
               void at::native::vectorized_elementwise_kernel<(i    12.2%   172.0
               void at::native::elementwise_kernel<(int)128, (in     6.8%    36.0
optimizer    void at::native::vectorized_elementwise_kernel<(i...   44.5%   666.0 否
     

- 用调用顺序还原名称

按发射线程里的 NVTX 路径归属（forward 和 optimizer）

这就是 nvtx_kern_sum 做的事。想拿到完整路径（step>forward>TransformerBlock>SwiGLU>Linear），用 sqlite 通过 correlationId 把 kernel 连回发射它的 CUDA API 调用，再按发射时间和线程找包含它的所有 NVTX 区间。

不要用 kernel 的 GPU 时间去落进 NVTX 区间。 你这个脚本对 forward/backward/optimizer 用 GPU 时间窗没问题，因为阶段末尾有 sync。但细粒度区间（Linear、attn.softmax）没有 sync，kernel 往往在 CPU 区间结束之后才执行，会被错归。

In [6]:
import re
RULES = [(r"direct_copy", "copy"), (r"where_kernel", "where(mask)"), (r"exp_kernel", "exp"),
         (r"neg_kernel", "neg"), (r"sqrt", "sqrt"), (r"MaxOps", "max-reduce"), (r"sum_functor", "sum-reduce"),
         (r"mean", "mean-reduce"), (r"CatArrayBatchedCopy", "cat"), (r"MulFunctor", "mul"),
         (r"CUDAFunctor_add|AddFunctor", "add"), (r"DivFunctor|div_true", "div"),
         (r"magma_sgemmEx", "cublas-sgemm")]
def label(n):
    s = re.sub(r"^void\s+", "", str(n))
    m = re.search(r"sgemm_(\d+x\d+)_\d+x\d+_(nn|nt|tn)", s)
    if m: return f"cutlass {m.group(1)} {m.group(2)}"
    op = next((v for p, v in RULES if re.search(p, s)), "other")
    kind = "vec" if "vectorized_elementwise" in s else ("elem" if "elementwise_kernel" in s else "")
    scalar = "(tensor⊙scalar)" if "BUnaryFunctor" in s else ""
    return f"{kind} {op}{scalar}".strip()

agg["label"] = agg["name"].map(label)

import sqlite3, pandas as pd
db = sqlite3.connect("../results/nsys/small_ctx512_fp32.sqlite")

k = pd.read_sql("""
  SELECT s.value AS name, k.start AS ks, k.end AS ke, r.start AS ls, r.globalTid AS tid
  FROM CUPTI_ACTIVITY_KIND_KERNEL k
  JOIN CUPTI_ACTIVITY_KIND_RUNTIME r ON k.correlationId = r.correlationId
  JOIN StringIds s ON k.demangledName = s.id""", db)

nv = pd.read_sql("""
  SELECT e.start, e.end, e.globalTid AS tid, COALESCE(e.text, s.value) AS name
  FROM NVTX_EVENTS e LEFT JOIN StringIds s ON e.textId = s.id
  WHERE e.end IS NOT NULL""", db)
nv["dur"] = nv.end - nv.start
by_tid = {t: g.sort_values("dur", ascending=False) for t, g in nv.groupby("tid")}

def path(ts, tid):
    g = by_tid.get(tid)
    if g is None: return ""
    return ">".join(g[(g.start <= ts) & (g.end >= ts)].name)     # 外层到内层

k["path"] = [path(a, b) for a, b in zip(k.ls, k.tid)]
k["leaf"] = k.path.str.split(">").str[-1]
k["dur"]  = k.ke - k.ks
k["label"] = k.name.map(label)

t = (k.groupby(["leaf", "label"]).agg(n=("dur", "size"), ms=("dur", lambda s: s.sum() / 1e6),
                                      avg_us=("dur", lambda s: s.mean() / 1e3)).reset_index())
t["n/step"] = t.n / 3; t["ms/step"] = t.ms / 3
print(t.sort_values("ms/step", ascending=False).head(30).round(2))

            leaf                   label     n     ms  avg_us  n/step  ms/step
32        Linear            cublas-sgemm   219  61.71  281.77    73.0    20.57
5                      cutlass 128x64 nt   216  42.00  194.44    72.0    14.00
4                      cutlass 128x64 nn   180  39.22  217.90    60.0    13.07
6                     cutlass 256x128 nn    75  37.98  506.36    25.0    12.66
21                               vec mul   669  35.64   53.27   223.0    11.88
33        Linear      cutlass 128x256 tn    36  20.44  567.65    12.0     6.81
11                              elem div   108  19.65  181.92    36.0     6.55
2                     cutlass 128x128 nt    39  19.06  488.72    13.0     6.35
19                               vec add   477  15.37   32.21   159.0     5.12
23                               vec neg    81  14.77  182.32    27.0     4.92
64     optimizer                 vec mul  1998  13.95    6.98   666.0     4.65
10                             elem copy   471  12.1

In [7]:
import sqlite3, re, pandas as pd
DB, STEPS = "../results/nsys/small_ctx512_fp32.sqlite", 3
B, T, d, h = 4, 512, 768, 12
db = sqlite3.connect(DB)

kern = pd.read_sql("""SELECT k.start AS ks, k.end AS ke, s.value AS name
    FROM CUPTI_ACTIVITY_KIND_KERNEL k JOIN StringIds s ON k.demangledName = s.id""", db)
kern["dur"] = kern.ke - kern.ks
nv = pd.read_sql("""SELECT e.start, e.end, e.globalTid AS tid, COALESCE(e.text, s.value) AS name
    FROM NVTX_EVENTS e LEFT JOIN StringIds s ON e.textId = s.id WHERE e.end IS NOT NULL""", db)
is_gemm = lambda n: any(x in n.lower() for x in ("gemm", "cutlass", "magma"))
kern["gemm"] = kern.name.map(is_gemm)

# A. 按阶段窗口归因（阶段末尾有 sync）
parts = []
for w in nv[nv.name.isin(["forward", "backward", "optimizer"])].itertuples():
    m = kern[(kern.ks >= w.start) & (kern.ke <= w.end)].copy(); m["phase"] = w.name; parts.append(m)
df = pd.concat(parts, ignore_index=True)

print("== 主表")
for ph in ["forward", "backward", "optimizer"]:
    s = df[df.phase == ph]; tot = s.dur.sum()
    g = s.groupby("name").agg(n=("dur", "size"), t=("dur", "sum")).sort_values("t", ascending=False)
    print(f"{ph:10s} {label(g.index[0]):32s} {100*g.t.iloc[0]/tot:5.1f}%  {g.n.iloc[0]/STEPS:6.1f}/步  "
          f"{'GEMM' if is_gemm(g.index[0]) else '非GEMM'}   阶段合计 {tot/1e6/STEPS:.2f} ms/步")

f  = df[df.phase == "forward"]; fb = df[df.phase.isin(["forward", "backward"])]
x  = f.groupby("name").dur.sum().idxmax()
print("\nQ2 同一 kernel：", label(x),
      f"forward {100*f[f.name==x].dur.sum()/f.dur.sum():.1f}% → fwd+bwd {100*fb[fb.name==x].dur.sum()/fb.dur.sum():.1f}%")
print(f"   GEMM 类：forward {100*f[f.gemm].dur.sum()/f.dur.sum():.1f}% → fwd+bwd {100*fb[fb.gemm].dur.sum()/fb.dur.sum():.1f}%"
      f" → 整步 {100*df[df.gemm].dur.sum()/df.dur.sum():.1f}%")

g = df.groupby("name").dur.sum().sort_values(ascending=False)
non = g[[not is_gemm(n) for n in g.index]]
print("Q3 非 GEMM 第一：", label(non.index[0]), f"{100*non.iloc[0]/df.dur.sum():.1f}%",
      "| elementwise 类", f"{100*df[df.name.str.contains('elementwise')].dur.sum()/df.dur.sum():.1f}%")

# B. attention 三段：靠发射线程上的 NVTX 区间（没有 sync，不能用 GPU 时间窗）
rt = pd.read_sql("""SELECT s.value AS name, k.start AS ks, k.end AS ke, r.start AS ls, r.globalTid AS tid
    FROM CUPTI_ACTIVITY_KIND_KERNEL k
    JOIN CUPTI_ACTIVITY_KIND_RUNTIME r ON k.correlationId = r.correlationId
    JOIN StringIds s ON k.demangledName = s.id""", db)
parts = []
for r in nv[nv.name.isin(["attn.scores", "attn.softmax", "attn.av"])].itertuples():
    m = rt[(rt.tid == r.tid) & (rt.ls >= r.start) & (rt.ls <= r.end)].copy(); m["sec"] = r.name; parts.append(m)
a = pd.concat(parts); a["dur"] = a.ke - a.ks
mm = a[a.name.map(is_gemm)].dur.sum() / STEPS / 1e6
sm = a[a.sec == "attn.softmax"].dur.sum() / STEPS / 1e6
fl_mm, fl_sm = 12 * 4 * B * T * T * d, 12 * 5 * B * h * T * T
print(f"\nQ5 耗时 softmax {sm:.2f} ms : matmul {mm:.2f} ms = {sm/mm:.2f} : 1")
print(f"   FLOPs softmax {fl_sm/1e9:.2f} G : matmul {fl_mm/1e9:.1f} G = {fl_sm/fl_mm:.3f} : 1")

== 主表
forward    cublas-sgemm                      47.5%    85.0/步  GEMM   阶段合计 46.14 ms/步
backward   cutlass 128x64 nt                 14.6%    72.0/步  GEMM   阶段合计 95.87 ms/步
optimizer  vec mul                           44.5%   666.0/步  非GEMM   阶段合计 10.44 ms/步

Q2 同一 kernel： cublas-sgemm forward 47.5% → fwd+bwd 16.6%
   GEMM 类：forward 65.7% → fwd+bwd 57.4% → 整步 53.5%
Q3 非 GEMM 第一： vec mul 8.7% | elementwise 类 43.2%

Q5 耗时 softmax 6.29 ms : matmul 2.93 ms = 2.15 : 1
   FLOPs softmax 0.75 G : matmul 38.7 G = 0.020 : 1


### 4.ncu测量

### 3.5 混合精度（bf16 autocast）
| 模型 | ctx | dtype | forward (ms) | backward (ms) | 峰值显存 (GB) | 加速比（相对 fp32） |
|---|---|---|---|---|---|---|
| small | 512 | fp32 |  |  |  | 1.00× |
| small | 512 | bf16 autocast |  |  |  |  |
| medium | 512 | fp32 |  |  |  | 1.00× |
| medium | 512 | bf16 autocast |  |  |  |  |
| large | 256 | fp32 |  |  |  | 1.00× |
| large | 256 | bf16 autocast |  |  |  |  |

附加：autocast 下各类算子的 dtype（作业必答）。

| 对象 | dtype |
|---|---|
| 模型参数 |  |
| 第一层 linear 输出 |  |
| LayerNorm / RMSNorm 输出 |  |
| logits |  |
| loss |  |
| 梯度 |  |

```bash
CUDA_VISIBLE_DEVICES=0 uv run python scripts/benchmark_v2.py --model small medium --ctx 512 --dtype float32 bfloat16
device: NVIDIA GeForce RTX 5070 Ti  torch 2.11.0+cu130  cuda 13.0  git 55d7ad1
提示：运行前请用 nvidia-smi 确认卡空闲；抖动(jitter)过大先查是否有别的进程。

[small ctx=512 b=4 float32] params=128.6M  matmul_precision=highest  status=OK  loss0=9.2808
  phase                mean±std    median      min      max  jitter
  fwd(no_grad)       37.62±0.08     37.65    37.43    37.68     1%   14.18 TFLOP/s  peak 0.71 GiB
  phase                mean±std    median      min      max  jitter
  forward(+loss)     38.56±0.04     38.55    38.51    38.66     0%
  backward           76.11±0.07     76.09    76.00    76.21     0%
  optimizer          13.07±0.06     13.04    12.98    13.18     2%
  step=sum          127.73±0.11    127.70   127.62   127.92     0%
  e2e wall 127.67 ms/步  16041 tok/s  12.54 TFLOP/s(3xF, 按 step 中位数)  peak 反向后 5.03 / 整步 5.03 GiB  loss有限=True

[small ctx=512 b=4 bfloat16] params=128.6M  matmul_precision=highest  status=OK  loss0=9.2808
  phase                mean±std    median      min      max  jitter
  fwd(no_grad)       18.76±0.01     18.76    18.74    18.77     0%   28.46 TFLOP/s  peak 0.89 GiB
  phase                mean±std    median      min      max  jitter
  forward(+loss)     20.18±0.02     20.18    20.14    20.21     0%
  backward           44.29±0.05     44.30    44.24    44.36     0%
  optimizer          13.03±0.04     13.04    12.96    13.10     1%
  step=sum           77.50±0.07     77.52    77.35    77.58     0%
  e2e wall 77.11 ms/步  26561 tok/s  20.66 TFLOP/s(3xF, 按 step 中位数)  peak 反向后 4.25 / 整步 4.25 GiB  loss有限=True

[medium ctx=512 b=4 float32] params=423.2M  matmul_precision=highest  status=OK  loss0=9.3120
  phase                mean±std    median      min      max  jitter
  fwd(no_grad)      112.18±0.04    112.19   112.08   112.22     0%   15.99 TFLOP/s  peak 1.89 GiB
  phase                mean±std    median      min      max  jitter
  forward(+loss)    113.01±0.11    113.02   112.84   113.19     0%
  backward          218.78±0.19    218.76   218.60   219.16     0%
  optimizer          47.69±0.38     47.71    47.21    48.18     2%
  step=sum          379.48±0.38    379.54   378.68   379.93     0%
  e2e wall 379.49 ms/步  5397 tok/s  14.18 TFLOP/s(3xF, 按 step 中位数)  peak 反向后 13.72 / 整步 13.72 GiB  loss有限=True

[medium ctx=512 b=4 bfloat16] params=423.2M  matmul_precision=highest  status=OK  loss0=9.3120
  phase                mean±std    median      min      max  jitter
  fwd(no_grad)       56.44±0.02     56.44    56.42    56.48     0%   31.79 TFLOP/s  peak 2.55 GiB
  phase                mean±std    median      min      max  jitter
  forward(+loss)     57.90±0.09     57.88    57.80    58.06     0%
  backward          126.74±0.07    126.73   126.63   126.92     0%
  optimizer          47.38±0.20     47.30    47.18    47.72     1%
  step=sum          232.01±0.17    231.99   231.73   232.23     0%
  e2e wall 231.61 ms/步  8842 tok/s  23.20 TFLOP/s(3xF, 按 step 中位数)  peak 反向后 11.62 / 整步 11.62 GiB  loss有限=True
```

### 3.2 端到端计时（ms，均值±样本std）
| 模型 | ctx | dtype | forward 均值±std (ms) | backward 均值±std (ms) | optimizer step 均值±std (ms) | 状态 |
|---|---|---|---|---|---|---|
| small | 512 | fp32 | 38.56±0.04 | 76.11±0.07 | 13.07±0.06 | OK |
| small | 512 | bf16 autocast | 20.18±0.02 | 44.29±0.05 | 13.03±0.04 | OK |
| medium | 512 | fp32 | 113.01±0.11 | 218.78±0.19 | 47.69±0.38 | OK |
| medium | 512 | bf16 autocast | 57.90±0.09 | 126.74±0.07 | 47.38±0.20 | OK |

说明：带 * 的是 no_grad 前向；其余 forward 是训练中的前向（含 loss、带计算图）。

### 3.5 混合精度（forward / backward 取训练步中的阶段均值；显存单位 GiB）
| 模型 | ctx | dtype | forward (ms) | backward (ms) | 峰值显存 (GiB) | 加速比（fwd+bwd，相对 fp32） | loss0 |
|---|---|---|---|---|---|---|---|
| small | 512 | fp32 | 38.56 | 76.11 | 5.03 | 1.00× | 9.2808 |
| small | 512 | bf16 autocast | 20.18 | 44.29 | 4.25 | 1.78× | 9.2808 |
| medium | 512 | fp32 | 113.01 | 218.78 | 13.72 | 1.00× | 9.3120 |
| medium | 512 | bf16 autocast | 57.90 | 126.74 | 11.62 | 1.80× | 9.3120 |

峰值显存：有优化器时取完整训练步，--no-opt 时取反向结束时。loss0 为刚初始化时的 loss，fp32 与 bf16 应非常接近。

saved -> results/bench/bench2_small_medium_ctx512_b4_float32-bfloat16.json

```bash
CUDA_VISIBLE_DEVICES=0 uv run python scripts/benchmark_v2.py --check-dtypes --model small --ctx 128 --dtype bfloat16
device: NVIDIA GeForce RTX 5070 Ti  torch 2.11.0+cu130  cuda 13.0  git 55d7ad1
提示：运行前请用 nvidia-smi 确认卡空闲；抖动(jitter)过大先查是否有别的进程。
```

### dtype 实测：small ctx=128 dtype=bfloat16
| 对象 | dtype |
|---|---|
| 模型参数 | float32 |
| 第一层 linear 输出 | bfloat16 |
| 所有 Linear 输出 | bfloat16 |
| LayerNorm / RMSNorm 输出 | float32 |
| TransformerBlock 输出（残差流） | float32 |
| logits | bfloat16 |
| loss | float32 |
| 梯度 | float32 |

### 3.6 attention 微基准（vanilla vs torch.compile）
固定 batch=8，单头，d_model ∈ {16, 32, 64, 128}，seq ∈ {256, 1024, 4096, 8192, 16384}。
> **理论预测（先写后测）**：backward 前需保存 scores + softmax 输出
> ≈ 2 × batch × S² × 4B。seq=16384 → ~17.2 GB → **预测 OOM 首现点**；
> seq=8192 → ~4.3 GB 可过。实测对照此预测。
**forward / backward 时间 (ms)，vanilla attention**

| d_model \ seq | 256 | 1024 | 4096 | 8192 | 16384 |
|---|---|---|---|---|---|
| 16 |  |  |  |  |  |
| 32 |  |  |  |  |  |
| 64 |  |  |  |  |  |
| 128 |  |  |  |  |  |

**backward 前显存 (GB)**（记录 OOM 的最小配置）

| d_model \ seq | 256 | 1024 | 4096 | 8192 | 16384 |
|---|---|---|---|---|---|
| 16 |  |  |  |  |  |
| 32 |  |  |  |  |  |
| 64 |  |  |  |  |  |
| 128 |  |  |  |  |  |

**torch.compile 对比（选 3 个代表配置）**

| d_model | seq | vanilla fwd | compiled fwd | vanilla bwd | compiled bwd | 加速比 |
|---|---|---|---|---|---|---|
|  |  |  |  |  |  |  |
|  |  |  |  |  |  |  |
|  |  |  |  |  |  |  |

**需要回答**：
- 最小 OOM 的配置是什么？显存主要来自哪个张量（N×N 的 score 矩阵）？
- 手算 backward 前保存的显存与实测是否吻合？
- 如何避免这个显存（引出 FlashAttention）？


- 3.6：vanilla 全网格（含 OOM 探测）+ 3 个代表配置的 compiled 对比

代表配置按瓶颈类型选：compute 主导(128,4096) / memory 主导(64,8192) / 贴极限(128,8192)

```bash
CUDA_VISIBLE_DEVICES=0 uv run python scripts/attn_microbench.py \
    --compile-cfgs 128,4096 64,8192 128,8192
```

```bash
CUDA_VISIBLE_DEVICES=0 uv run python scripts/attn_microbench.py     --compile-cfgs 128,4096 64,8192 128,8192
device: NVIDIA GeForce RTX 5070 Ti  b=8 单头 fp32(TF32 off)
理论 forward 瞬态峰值 2·B·S²·4B：S=256:0.00GiB  S=1024:0.06GiB  S=4096:1.00GiB  S=8192:4.00GiB  S=16384:16.00GiB
理论 backward 前保存  1·B·S²·4B：S=256:0.00GiB  S=1024:0.03GiB  S=4096:0.50GiB  S=8192:2.00GiB  S=16384:8.00GiB
  vanilla  d=  16 S=   256: fwd=    0.16 bwd=    0.34 pre=  0.02G peakF=  0.02G
  vanilla  d=  32 S=   256: fwd=    0.17 bwd=    0.34 pre=  0.02G peakF=  0.02G
  vanilla  d=  64 S=   256: fwd=    0.16 bwd=    0.33 pre=  0.02G peakF=  0.02G
  vanilla  d= 128 S=   256: fwd=    0.06 bwd=    0.26 pre=  0.02G peakF=  0.02G
  vanilla  d=  16 S=  1024: fwd=    0.20 bwd=    0.44 pre=  0.05G peakF=  0.08G
  vanilla  d=  32 S=  1024: fwd=    0.20 bwd=    0.44 pre=  0.05G peakF=  0.08G
  vanilla  d=  64 S=  1024: fwd=    0.34 bwd=    0.49 pre=  0.05G peakF=  0.09G
  vanilla  d= 128 S=  1024: fwd=    0.52 bwd=    0.79 pre=  0.06G peakF=  0.10G
  vanilla  d=  16 S=  4096: fwd=    4.76 bwd=   11.41 pre=  0.52G peakF=  1.03G
  vanilla  d=  32 S=  4096: fwd=    4.85 bwd=    9.61 pre=  0.53G peakF=  1.04G
  vanilla  d=  64 S=  4096: fwd=    6.46 bwd=   10.77 pre=  0.55G peakF=  1.05G
  vanilla  d= 128 S=  4096: fwd=    9.57 bwd=   16.05 pre=  0.58G peakF=  1.09G
  vanilla  d=  16 S=  8192: fwd=   25.06 bwd=   41.53 pre=  2.03G peakF=  4.04G
  vanilla  d=  32 S=  8192: fwd=   25.23 bwd=   41.63 pre=  2.05G peakF=  4.05G
  vanilla  d=  64 S=  8192: fwd=   29.41 bwd=   41.12 pre=  2.08G peakF=  4.09G
  vanilla  d= 128 S=  8192: fwd=   37.56 bwd=   68.40 pre=  2.14G peakF=  4.17G
  vanilla  d=  16 S= 16384: OOM
  vanilla  d=  32 S= 16384: OOM
  vanilla  d=  64 S= 16384: OOM
  vanilla  d= 128 S= 16384: OOM
/data2/c220/hz/cs336/.venv/lib/python3.13/site-packages/torch/_inductor/compile_fx.py:322: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(
  compiled d= 128 S=  4096: fwd=    6.30 bwd=   10.70 pre=  0.58G peakF=  0.59G   (含编译共 7s)
  compiled d=  64 S=  8192: fwd=   21.07 bwd=   26.21 pre=  2.08G peakF=  2.09G   (含编译共 2s)
  compiled d= 128 S=  8192: fwd=   31.30 bwd=   51.55 pre=  2.14G peakF=  2.17G   (含编译共 1s)
```

**forward / backward 时间 (ms)，vanilla attention（格内为 fwd / bwd）**
| d_model \ seq | 256 | 1024 | 4096 | 8192 | 16384 |
|---|---|---|---|---|---|
| 16 | 0.16 / 0.34 | 0.20 / 0.44 | 4.76 / 11.41 | 25.06 / 41.53 | OOM |
| 32 | 0.17 / 0.34 | 0.20 / 0.44 | 4.85 / 9.61 | 25.23 / 41.63 | OOM |
| 64 | 0.16 / 0.33 | 0.34 / 0.49 | 6.46 / 10.77 | 29.41 / 41.12 | OOM |
| 128 | 0.06 / 0.26 | 0.52 / 0.79 | 9.57 / 16.05 | 37.56 / 68.40 | OOM |

**backward 前显存 (GiB)，vanilla —— 对账基准 1·B·S²·4B**
| d_model \ seq | 256 | 1024 | 4096 | 8192 | 16384 |
|---|---|---|---|---|---|
| 16 | 0.02 | 0.05 | 0.52 | 2.03 | OOM |
| 32 | 0.02 | 0.05 | 0.53 | 2.05 | OOM |
| 64 | 0.02 | 0.05 | 0.55 | 2.08 | OOM |
| 128 | 0.02 | 0.06 | 0.58 | 2.14 | OOM |

**[附加] forward 瞬态峰值显存 (GiB) —— 对账基准 2·B·S²·4B，OOM 咬人的位置**
| d_model \ seq | 256 | 1024 | 4096 | 8192 | 16384 |
|---|---|---|---|---|---|
| 16 | 0.02 | 0.08 | 1.03 | 4.04 | OOM |
| 32 | 0.02 | 0.08 | 1.04 | 4.05 | OOM |
| 64 | 0.02 | 0.09 | 1.05 | 4.09 | OOM |
| 128 | 0.02 | 0.10 | 1.09 | 4.17 | OOM |

**torch.compile 对比**
| d_model | seq | vanilla fwd | compiled fwd | vanilla bwd | compiled bwd | 加速比(fwd+bwd) | vanilla pre-G | compiled pre-G |
|---|---|---|---|---|---|---|---|---|
| 128 | 4096 | 9.57 | 6.30 | 16.05 | 10.70 | 1.51× | 0.58 | 0.58 |
| 64 | 8192 | 29.41 | 21.07 | 41.12 | 26.21 | 1.49× | 2.08 | 2.08 |
| 128 | 8192 | 37.56 | 31.30 | 68.40 | 51.55 | 1.28× | 2.14 | 2.14 |

### 3.7 完整模型 compile

| 模型 | ctx | vanilla fwd | compiled fwd | vanilla fwd+bwd | compiled fwd+bwd | vanilla 全步 | compiled 全步 |
|---|---|---|---|---|---|---|---|
| small | 512 |  |  |  |  |  |  |
| medium | 512 |  |  |  |  |  |  |

注意：compile 首次调用含编译时间，warmup 必须排除。

- 3.7：vanilla 与 compiled 各跑一次（唯一变量 = --compile；compiled 的 warmup 加大以吸收编译）

```bash
CUDA_VISIBLE_DEVICES=0 uv run python scripts/benchmark_v2.py \
    --model small medium --ctx 512 --dtype float32 --mode train --warmup 5 --iters 10
CUDA_VISIBLE_DEVICES=0 uv run python scripts/benchmark_v2.py \
    --model small medium --ctx 512 --dtype float32 --mode train --warmup 8 --iters 10 --compile
```

```bash
CUDA_VISIBLE_DEVICES=0 uv run python scripts/benchmark_v2.py \
    --model small medium --ctx 512 --dtype float32 --mode train --warmup 5 --iters 10
device: NVIDIA GeForce RTX 5070 Ti  torch 2.11.0+cu130  cuda 13.0  git 55d7ad1
提示：运行前请用 nvidia-smi 确认卡空闲；抖动(jitter)过大先查是否有别的进程。

[small ctx=512 b=4 float32] params=128.6M  matmul_precision=highest  status=OK  loss0=9.2808
  phase                mean±std    median      min      max  jitter
  forward(+loss)     52.93±0.26     52.96    52.58    53.44     2%
  backward           98.53±2.47     98.59    95.96   100.99     5%
  optimizer          15.37±1.52     15.37    13.76    16.97    21%
  step=sum          166.82±3.76    166.74   162.92   170.78     5%
  e2e wall 166.07 ms/步  12332 tok/s  9.61 TFLOP/s(3xF, 按 step 中位数)  peak 反向后 5.03 / 整步 5.03 GiB  loss有限=True

[medium ctx=512 b=4 float32] params=423.2M  matmul_precision=highest  status=OOM(train)  loss0=9.3120
```

### 3.2 端到端计时（ms，均值±样本std）
| 模型 | ctx | dtype | forward 均值±std (ms) | backward 均值±std (ms) | optimizer step 均值±std (ms) | 状态 |
|---|---|---|---|---|---|---|
| small | 512 | fp32 | 52.93±0.26 | 98.53±2.47 | 15.37±1.52 | OK |
| medium | 512 | fp32 | N/A | N/A | N/A | OOM(train) |

说明：带 * 的是 no_grad 前向；其余 forward 是训练中的前向（含 loss、带计算图）。

saved -> results/bench/bench2_small_medium_ctx512_b4_float32.json

```bash
CUDA_VISIBLE_DEVICES=0 uv run python scripts/benchmark_v2.py     --model small medium --ctx 512 --dtype float32 --mode train --warmup 8 --iters 10 --com
pile
device: NVIDIA GeForce RTX 5070 Ti  torch 2.11.0+cu130  cuda 13.0  git 55d7ad1
提示：运行前请用 nvidia-smi 确认卡空闲；抖动(jitter)过大先查是否有别的进程。
/data2/c220/hz/cs336/.venv/lib/python3.13/site-packages/torch/_inductor/compile_fx.py:322: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(

[small ctx=512 b=4 float32] params=128.6M  matmul_precision=highest  status=OK  loss0=9.2808
  phase                mean±std    median      min      max  jitter
  forward(+loss)     38.98±2.62     39.97    34.13    41.45    18%
  backward           69.69±0.13     69.68    69.43    69.92     1%
  optimizer          16.73±1.89     16.82    13.80    20.02    37%
  step=sum          125.41±2.21    126.31   120.78   127.03     5%
  e2e wall 125.01 ms/步  16383 tok/s  12.68 TFLOP/s(3xF, 按 step 中位数)  peak 反向后 4.32 / 整步 4.32 GiB  loss有限=True

[medium ctx=512 b=4 float32] params=423.2M  matmul_precision=highest  status=OK  loss0=9.3120
  phase                mean±std    median      min      max  jitter
  forward(+loss)    110.74±5.44    113.23   102.77   116.12    12%
  backward          197.38±9.51    201.54   186.49   206.42    10%
  optimizer          56.89±3.40     56.10    52.97    63.12    18%
  step=sum         365.01±17.72    372.00   344.13   384.60    11%
  e2e wall 345.73 ms/步  5924 tok/s  14.47 TFLOP/s(3xF, 按 step 中位数)  peak 反向后 11.83 / 整步 11.83 GiB  loss有限=True
```
### 3.2 端到端计时（ms，均值±样本std）
| 模型 | ctx | dtype | forward 均值±std (ms) | backward 均值±std (ms) | optimizer step 均值±std (ms) | 状态 |
|---|---|---|---|---|---|---|
| small | 512 | fp32 | 38.98±2.62 | 69.69±0.13 | 16.73±1.89 | OK |
| medium | 512 | fp32 | 110.74±5.44 | 197.38±9.51 | 56.89±3.40 | OK |

说明：带 * 的是 no_grad 前向；其余 forward 是训练中的前向（含 loss、带计算图）。

saved -> results/bench/bench2_small_medium_ctx512_b4_float32_compile.json

- 拼 3.7 表（文件名改）

In [1]:
#!/usr/bin/env python
"""3.7 完整模型 compile 表：把 v2 benchmark 跑出的 vanilla / compiled 两份 JSON 拼表。
思路：3.7 复用 12.2 的分阶段口径（train 模式），唯一变量是 --compile 开关，
因此不需要新测量代码，只需要聚合。三列加速比递减本身就是发现：
  fwd 列        compile 管得到（fusion 逐元素链）
  fwd+bwd 列    bwd 图也被 inductor 编译（aot_autograd），仍管得到
  全步列        optimizer 段是自定义 AdamW 的逐元素 op，compile 管不到 → 稀释加速比
"""
import json
import sys
from pathlib import Path

p1= "../results/bench/bench2_small_medium_ctx512_b4_float32.json"
p2= "../results/bench/bench2_small_medium_ctx512_b4_float32_compile.json"

def load(p):
    return json.loads(Path(p).read_text())


def index(doc):
    """按 (model, ctx) 建索引，只收 train 成功的配置。"""
    out = {}
    for r in doc["results"]:
        if r.get("train") and r["status"].get("train") == "OK":
            out[(r["model"], r["ctx"])] = r
    return out


def main():
    van, comp = index(load(p1)), index(load(p2))
    keys = [k for k in van if k in comp]
    print("\n### 3.7 完整模型 compile（ms，均值口径；加速比为全步）")
    print("| 模型 | ctx | vanilla fwd | compiled fwd | vanilla fwd+bwd | compiled fwd+bwd "
          "| vanilla 全步 | compiled 全步 | 全步加速比 |")
    print("|---|---|---|---|---|---|---|---|---|")
    for m, c in keys:
        tv, tc = van[(m, c)]["train"], comp[(m, c)]["train"]
        vf, cf = tv["forward"]["mean"], tc["forward"]["mean"]
        vb, cb = tv["backward"]["mean"], tc["backward"]["mean"]
        vs, cs = tv["step"]["mean"], tc["step"]["mean"]
        # 全步加速比：分母含 optimizer 段（compile 不加速）→ 天然低于 fwd 列的比值
        sp = vs / cs
        print(f"| {m} | {c} | {vf:.2f} | {cf:.2f} | {vf + vb:.2f} | {cf + cb:.2f} "
              f"| {vs:.2f} | {cs:.2f} | {sp:.2f}× |")
        # 编译泄漏检查：compiled 的第一个测量步若显著慢于中位数，说明编译没被 warmup 吃干净
        raw0 = tc["forward"]["raw"][0]
        med = tc["forward"]["median"]
        if raw0 > 1.5 * med:
            print(f"  WARNING: compiled forward 首步 {raw0:.2f}ms > 1.5×median，"
                  f"warmup 可能不够，编译时间泄漏进了测量区")
    print("\n注意：fwd 为训练步中带图前向段（含 loss），与 fwd(no_grad) 口径不同，勿混填。")

main()


### 3.7 完整模型 compile（ms，均值口径；加速比为全步）
| 模型 | ctx | vanilla fwd | compiled fwd | vanilla fwd+bwd | compiled fwd+bwd | vanilla 全步 | compiled 全步 | 全步加速比 |
|---|---|---|---|---|---|---|---|---|
| small | 512 | 54.55 | 38.98 | 157.84 | 108.68 | 173.32 | 125.41 | 1.38× |

注意：fwd 为训练步中带图前向段（含 loss），与 fwd(no_grad) 口径不同，勿混填。


- 3.7 完整模型 compile（ms，均值口径；加速比为全步）

| 模型 | ctx | vanilla fwd | compiled fwd | vanilla fwd+bwd | compiled fwd+bwd | vanilla 全步 | compiled 全步 | 全步加速比 |
|---|---|---|---|---|---|---|---|---|
| small | 512 | 54.55 | 38.98 | 157.84 | 108.68 | 173.32 | 125.41 | 1.38× |

注意：fwd 为训练步中带图前向段（含 loss），与 fwd(no_grad) 口径不同，勿混填。

### 3.8 显存 profiling
方法：`torch.cuda.memory._record_memory_history(max_entries=...)`，跑完导出
pickle，用 pytorch.org/memory_viz 查看。
| 模型 | ctx | dtype | forward 峰值 (GB) | 完整训练步峰值 (GB) | 理论静态显存 (GB) | 主要占用来源 |
|---|---|---|---|---|---|---|
| medium | 128 | fp32 |  |  |  |  |
| medium | 512 | fp32 |  |  |  |  |
| medium | 128 | bf16 |  |  |  |  |
| medium | 512 | bf16 |  |  |  |  |
| large | 256 | fp32 |  |  |  |  |
| large | 256 | bf16 |  |  |  |  |

附加：
| 问题 | 答案 |
|---|---|
| 激活值显存与 ctx 的关系？ |  |
| 2.7B 理论峰值（参数+梯度+Adam+激活估算）？ |  |
| 最大的单次分配（如 attention score）是多大？ |  |
| 混合精度对峰值显存的影响？ |  |

- 详见profile_3_mem.md

```bash
CUDA_VISIBLE_DEVICES=0 uv run python scripts/mem_profile.py --model medium --ctx 128 256 512 --dtype float32 bfloat16

=== medium ctx=128 b=4 float32 ===
理论：P=423.2M  4P=1.58  16P=6.31  激活估算=2.14  峰值估算=8.44 GiB  (显卡 15.5 GiB，放得下)
A no_grad 前向峰值 1.64 GiB | B 保存的激活 1.64 GiB | C 完整步峰值 6.52 GiB | status=OK

=== medium ctx=128 b=4 bfloat16 ===
理论：P=423.2M  4P=1.58  16P=6.31  激活估算=2.14  峰值估算=8.44 GiB  (显卡 15.5 GiB，放得下)
A no_grad 前向峰值 2.37 GiB | B 保存的激活 1.82 GiB | C 完整步峰值 6.62 GiB | status=OK

=== medium ctx=256 b=4 float32 ===
理论：P=423.2M  4P=1.58  16P=6.31  激活估算=4.65  峰值估算=10.96 GiB  (显卡 15.5 GiB，放得下)
A no_grad 前向峰值 1.70 GiB | B 保存的激活 3.66 GiB | C 完整步峰值 8.49 GiB | status=OK

=== medium ctx=256 b=4 bfloat16 ===
理论：P=423.2M  4P=1.58  16P=6.31  激活估算=4.65  峰值估算=10.96 GiB  (显卡 15.5 GiB，放得下)
A no_grad 前向峰值 2.39 GiB | B 保存的激活 3.16 GiB | C 完整步峰值 8.01 GiB | status=OK

=== medium ctx=512 b=4 float32 ===
理论：P=423.2M  4P=1.58  16P=6.31  激活估算=10.81  峰值估算=17.11 GiB  (显卡 15.5 GiB，预计放不下（OOM 是预期结果）)
A no_grad 前向峰值 1.88 GiB | B 保存的激活 8.82 GiB | C 完整步峰值 13.72 GiB | status=OK

=== medium ctx=512 b=4 bfloat16 ===
理论：P=423.2M  4P=1.58  16P=6.31  激活估算=10.81  峰值估算=17.11 GiB  (显卡 15.5 GiB，预计放不下（OOM 是预期结果）)
A no_grad 前向峰值 2.54 GiB | B 保存的激活 6.68 GiB | C 完整步峰值 11.62 GiB | status=OK
```


============================== 汇总 ==============================

### 3.8 显存 profiling（单位 GiB）
| 模型 | ctx | dtype | forward 峰值(no_grad) | 带图 forward 峰值 | 完整训练步峰值 | 峰值阶段 | 理论 4P | 理论 16P | 峰值估算(11.6) | 估算÷实测 | 状态 |
|---|---|---|---|---|---|---|---|---|---|---|---|
| medium | 128 | fp32 | 1.64 | 3.27 | 6.52 | optimizer | 1.58 | 6.31 | 8.44 | 1.29 | OK |
| medium | 128 | bf16 autocast | 2.37 | 3.46 | 6.62 | forward | 1.58 | 6.31 | 8.44 | 1.27 | OK |
| medium | 256 | fp32 | 1.70 | 5.32 | 8.49 | forward | 1.58 | 6.31 | 10.96 | 1.29 | OK |
| medium | 256 | bf16 autocast | 2.39 | 4.84 | 8.01 | forward | 1.58 | 6.31 | 10.96 | 1.37 | OK |
| medium | 512 | fp32 | 1.88 | 10.56 | 13.72 | forward | 1.58 | 6.31 | 17.11 | 1.25 | OK |
| medium | 512 | bf16 autocast | 2.54 | 8.45 | 11.62 | forward | 1.58 | 6.31 | 17.11 | 1.47 | OK |

### 显存构成（GiB）：forward 结束时 与 backward 结束时
| 模型 | ctx | dtype | fwd结束: 权重 | Adam | 激活/其他 | 合计 | bwd结束: 权重 | 梯度 | Adam | 其他 | 合计 |
|---|---|---|---|---|---|---|---|---|---|---|---|
| medium | 128 | fp32 | 1.58 | 3.15 | 1.66 | 6.39 | 1.58 | 1.58 | 3.15 | 0.02 | 6.33 |
| medium | 128 | bf16 autocast | 1.58 | 3.15 | 1.85 | 6.57 | 1.58 | 1.58 | 3.15 | 0.02 | 6.33 |
| medium | 256 | fp32 | 1.58 | 3.15 | 3.68 | 8.41 | 1.58 | 1.58 | 3.15 | 0.02 | 6.33 |
| medium | 256 | bf16 autocast | 1.58 | 3.15 | 3.18 | 7.91 | 1.58 | 1.58 | 3.15 | 0.02 | 6.33 |
| medium | 512 | fp32 | 1.58 | 3.15 | 8.84 | 13.57 | 1.58 | 1.58 | 3.15 | 0.02 | 6.33 |
| medium | 512 | bf16 autocast | 1.58 | 3.15 | 6.70 | 11.43 | 1.58 | 1.58 | 3.15 | 0.02 | 6.33 |

说明：forward 结束时的『激活/其他』≈ 保存给反向用的激活（autocast 下含缓存的 bf16 权重副本）。

### 快照解析：峰值时刻的存活块归类（启发式，GiB）与最大单次分配

- medium ctx=128 fp32  事件数 19227  事件流峰值 6.52 GiB（allocator 口径 6.52，二者应接近）
  归类：静态(权重+Adam) 4.75；backward(梯度/临时) 1.58；optimizer 临时 0.19
  最大分配     39.1 MiB  ??:0 torch::unwind::unwind() <- ??:0 torch::CapturedTraceback::gather(bool, bool, bool) <- memory_snapshot.cpp:0 torch::cuda::(anonymous namespace)::gather_with_cpp()
  最大分配     39.1 MiB  ??:0 torch::unwind::unwind() <- ??:0 torch::CapturedTraceback::gather(bool, bool, bool) <- memory_snapshot.cpp:0 torch::cuda::(anonymous namespace)::gather_with_cpp()
  最大分配     39.1 MiB  ??:0 torch::unwind::unwind() <- ??:0 torch::CapturedTraceback::gather(bool, bool, bool) <- memory_snapshot.cpp:0 torch::cuda::(anonymous namespace)::gather_with_cpp()

- medium ctx=128 bf16 autocast  事件数 22230  事件流峰值 6.62 GiB（allocator 口径 6.62，二者应接近）
  归类：静态(权重+Adam) 4.75；backward(梯度/临时) 1.87
  最大分配     39.1 MiB  ??:0 torch::unwind::unwind() <- ??:0 torch::CapturedTraceback::gather(bool, bool, bool) <- memory_snapshot.cpp:0 torch::cuda::(anonymous namespace)::gather_with_cpp()
  最大分配     39.1 MiB  ??:0 torch::unwind::unwind() <- ??:0 torch::CapturedTraceback::gather(bool, bool, bool) <- memory_snapshot.cpp:0 torch::cuda::(anonymous namespace)::gather_with_cpp()
  最大分配     39.1 MiB  ??:0 torch::unwind::unwind() <- ??:0 torch::CapturedTraceback::gather(bool, bool, bool) <- memory_snapshot.cpp:0 torch::cuda::(anonymous namespace)::gather_with_cpp()

- medium ctx=256 fp32  事件数 19650  事件流峰值 8.48 GiB（allocator 口径 8.49，二者应接近）
  归类：静态(权重+Adam) 4.75；backward(梯度/临时) 3.73
  最大分配     39.1 MiB  ??:0 torch::unwind::unwind() <- ??:0 torch::CapturedTraceback::gather(bool, bool, bool) <- memory_snapshot.cpp:0 torch::cuda::(anonymous namespace)::gather_with_cpp()
  最大分配     39.1 MiB  ??:0 torch::unwind::unwind() <- ??:0 torch::CapturedTraceback::gather(bool, bool, bool) <- memory_snapshot.cpp:0 torch::cuda::(anonymous namespace)::gather_with_cpp()
  最大分配     39.1 MiB  ??:0 torch::unwind::unwind() <- ??:0 torch::CapturedTraceback::gather(bool, bool, bool) <- memory_snapshot.cpp:0 torch::cuda::(anonymous namespace)::gather_with_cpp()

- medium ctx=256 bf16 autocast  事件数 22501  事件流峰值 8.00 GiB（allocator 口径 8.01，二者应接近）
  归类：静态(权重+Adam) 4.75；backward(梯度/临时) 3.25
  最大分配     39.1 MiB  ??:0 torch::unwind::unwind() <- ??:0 torch::CapturedTraceback::gather(bool, bool, bool) <- memory_snapshot.cpp:0 torch::cuda::(anonymous namespace)::gather_with_cpp()
  最大分配     39.1 MiB  ??:0 torch::unwind::unwind() <- ??:0 torch::CapturedTraceback::gather(bool, bool, bool) <- memory_snapshot.cpp:0 torch::cuda::(anonymous namespace)::gather_with_cpp()
  最大分配     39.1 MiB  ??:0 torch::unwind::unwind() <- ??:0 torch::CapturedTraceback::gather(bool, bool, bool) <- memory_snapshot.cpp:0 torch::cuda::(anonymous namespace)::gather_with_cpp()

- medium ctx=512 fp32  事件数 19642  事件流峰值 13.72 GiB（allocator 口径 13.72，二者应接近）
  归类：backward(梯度/临时) 8.97；静态(权重+Adam) 4.75
  最大分配     78.1 MiB  ??:0 torch::unwind::unwind() <- ??:0 torch::CapturedTraceback::gather(bool, bool, bool) <- memory_snapshot.cpp:0 torch::cuda::(anonymous namespace)::gather_with_cpp()
  最大分配     78.1 MiB  ??:0 torch::unwind::unwind() <- ??:0 torch::CapturedTraceback::gather(bool, bool, bool) <- memory_snapshot.cpp:0 torch::cuda::(anonymous namespace)::gather_with_cpp()
  最大分配     78.1 MiB  ??:0 torch::unwind::unwind() <- ??:0 torch::CapturedTraceback::gather(bool, bool, bool) <- memory_snapshot.cpp:0 torch::cuda::(anonymous namespace)::gather_with_cpp()

- medium ctx=512 bf16 autocast  事件数 22519  事件流峰值 11.62 GiB（allocator 口径 11.62，二者应接近）
  归类：backward(梯度/临时) 6.87；静态(权重+Adam) 4.75
  最大分配     78.1 MiB  ??:0 torch::unwind::unwind() <- ??:0 torch::CapturedTraceback::gather(bool, bool, bool) <- memory_snapshot.cpp:0 torch::cuda::(anonymous namespace)::gather_with_cpp()
  最大分配     78.1 MiB  ??:0 torch::unwind::unwind() <- ??:0 torch::CapturedTraceback::gather(bool, bool, bool) <- memory_snapshot.cpp:0 torch::cuda::(anonymous namespace)::gather_with_cpp()
  最大分配     78.1 MiB  ??:0 torch::unwind::unwind() <- ??:0 torch::CapturedTraceback::gather(bool, bool, bool) <- memory_snapshot.cpp:0 torch::cuda::(anonymous namespace)::gather_with_cpp()

### 附加问题 1：激活显存与 ctx 的关系（带图前向后保存的激活，GiB）
- medium fp32: ctx128=1.64  ctx256=3.66  ctx512=8.82
  ctx 增大 4×：激活增大 5.4×（纯线性项应为 4×，纯 T² 项为 16×）
  拟合 act = a·N + b·B·h·T²：每层线性项等效 29.4 份 N×d 张量（11.6 取约 40），T² 项等效 c = 2.00（11.6 取 2）
- medium bf16 autocast: ctx128=1.82  ctx256=3.16  ctx512=6.68
  ctx 增大 4×：激活增大 3.7×（纯线性项应为 4×，纯 T² 项为 16×）
  拟合 act = a·N + b·B·h·T²：每层线性项等效 33.2 份 N×d 张量（11.6 取约 40），T² 项等效 c = 0.08（11.6 取 2）

### 附加问题 2：2.7B 理论峰值（用本配置的参数个数）
- P = 3.41B（SwiGLU 三矩阵、词表 10000，不是名字里的 2.7B）
- batch=4, ctx=128：静态 16P = 50.8 GiB；激活估算（c=2.00, other=9.4）= 5.2 GiB；合计 ≈ 55.9 GiB（60.1 GB）
- 仅推理：fp32 权重 4P = 12.7 GiB；纯 bf16 权重 2P = 6.3 GiB

### 附加问题 4：混合精度对峰值显存的影响（bf16 autocast ÷ fp32）
- medium ctx=128: 保存的激活 1.11×，no_grad 前向峰值 1.44×，完整步峰值 1.02×
- medium ctx=256: 保存的激活 0.86×，no_grad 前向峰值 1.40×，完整步峰值 0.94×
- medium ctx=512: 保存的激活 0.76×，no_grad 前向峰值 1.35×，完整步峰值 0.85×

saved -> results/mem/summary.json